# Prepare E03 caption features once for both VLMs

CPU only. Use a clean published E03 checkout and the shared plan printed by
`prepare_stage2.ipynb`. Mount the intended Drive account before setup. If needed,
install `requirements-stage2-cpu.txt` in this CPU runtime. No model selector,
GPU packages, weights or Hugging Face authentication are needed.

This produces 13 fixed preprocessing families. It fits vocabulary/IDF on FIT only,
preserves original captions and all zero-feature rows, and never processes CONFIRM.
It does not fit heads or produce scientific scores. See `docs/stage2_e03.md`.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO = Path('/content/Research')  # Published E03 implementation checkout.
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
if not (REPO/'src/stage2/prepare_e03.py').is_file():
    raise ValueError('Set REPO to the E03 implementation checkout')
if not PHASE1_ROOT.is_dir():
    raise ValueError('Mount the intended Drive account and set PHASE1_ROOT')
os.chdir(REPO)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
# Rerun settings to replace all prior settings. Every execution revalidates in a fresh process.
SHARED_PLAN = 'REPLACE_WITH_SHARED_PLAN/_COMPLETE.json'
RESUME_INDEX = None  # Explicit E03 checkpoint _COMPLETE.json printed by a prior run.
THROUGH_SPLIT = 'DEV-S'  # Publish all 13 FIT/DEV-S families promptly.
# To finish later: set RESUME_INDEX to the printed checkpoint and THROUGH_SPLIT='SCREEN'.
STORAGE_SCOPE = 'persistent-drive'


In [ ]:
# Fail before fitting on missing paths, placeholders, dirty code or incompatible plans/runtime.
from src.stage2.schema import marker_path
if 'REPLACE_' in str(SHARED_PLAN):
    raise ValueError('Supply the shared plan printed by prepare_stage2.ipynb')
for value in (SHARED_PLAN, RESUME_INDEX):
    if value is not None and not marker_path(value, PHASE1_ROOT).is_file():
        raise ValueError('Missing explicit artifact: '+str(value))
command = [sys.executable,'-m','src.stage2.prepare_e03', 'preflight',
           '--phase1-root',str(PHASE1_ROOT),'--manifest',str(SHARED_PLAN)]
subprocess.run(command,check=True)


In [ ]:
# This cell constructs fresh arguments from current settings; it never trusts a stale preflight flag.
command = [sys.executable,'-m','src.stage2.prepare_e03','run',
           '--phase1-root',str(PHASE1_ROOT),'--manifest',str(SHARED_PLAN),
           '--through-split',THROUGH_SPLIT,'--storage-scope',STORAGE_SCOPE]
if RESUME_INDEX is not None:
    command += ['--resume-index',str(RESUME_INDEX)]
print(json.dumps({'shared_plan':str(SHARED_PLAN),'resume_index':str(RESUME_INDEX),
                  'through_split':THROUGH_SPLIT,'phase1_root':str(PHASE1_ROOT)},indent=2))
# Stream checkpoint references as they are committed. Preserve the last checkpoint
# line if interrupted; completed artifacts are immutable and no directory is auto-discovered.
with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    for line in process.stdout:
        print(line,end='',flush=True)
    status=process.wait()
if status:
    raise RuntimeError('E03 stopped; inspect the diagnostic and retain the last checkpoint reference')


Each `release_passed` follows independent-process reconstruction and exact reload
comparison. The `checkpoint` event supplies an immutable resume index and family
completion references. Partial families can be consumed under the corresponding
FIT/DEV-S reader role; DEV-C/SCREEN remain separate releases. Use the same artifacts
for Qwen and Gemma. A finished E03 run does not establish whole-stage completion,
remote Drive sync, G0 completion or scientific results.
